# M5 Forecasting: Data Quality and Exploratory Data Analysis (EDA)

In this notebook, we perform the initial data contract validation and exploratory data analysis of the Walmart sales dataset. Our analysis targets the scoped hierarchy: `state_id = CA` and `cat_id = FOODS`.

In [ ]:
import json
import os

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. Load Data Quality Audit Report
quality_report_path = '../reports/data_quality.json'
if os.path.exists(quality_report_path):
    with open(quality_report_path, 'r') as f:
        quality_report = json.load(f)
    print("=== Data Quality Audit Results ===")
    print(json.dumps(quality_report, indent=4))
else:
    print("Data quality report not found at reports/data_quality.json!")

## 2. Load Data from Feature Mart (DuckDB)

In [ ]:
# Connect to DuckDB and load the analytic mart
db_path = '../data/processed/smart_replenishment.db'
con = duckdb.connect(db_path)
df_mart = con.execute("SELECT * FROM feature_mart LIMIT 500000").df()
con.close()

df_mart['date'] = pd.to_datetime(df_mart['date'])
print(f"Loaded subset size: {df_mart.shape}")
df_mart.head()

## 3. Exploratory Visualizations

### A. Overall Demand Over Time with Calendar Events
We aggregate demand by date and plot it along with calendar events to see if holidays or events impact sales.

In [ ]:
daily_demand = df_mart.groupby('date')['demand'].sum().reset_index()

plt.figure(figsize=(14, 5))
plt.plot(daily_demand['date'], daily_demand['demand'], color='royalblue', label='Daily Demand')
plt.title('Daily Demand Dynamics (CA State, FOODS category)')
plt.xlabel('Date')
plt.ylabel('Total Sales Units')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### B. Seasonality Analysis
Let's look at weekly (day of week) and monthly seasonality of demand.

In [ ]:
# Day of week seasonality
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_mart, x='wday', y='demand', showfliers=False, palette='crest')
plt.title('Demand Distribution by Day of Week (wday 1=Saturday, 2=Sunday)')
plt.xlabel('Day of Week Index')
plt.ylabel('Demand Units')
plt.show()

# Monthly seasonality
plt.figure(figsize=(10, 4))
sns.barplot(data=df_mart, x='month', y='demand', palette='viridis')
plt.title('Average Demand by Month')
plt.xlabel('Month')
plt.ylabel('Mean Demand')
plt.show()

### C. Intermittency (Zero Demand Distribution)
For retail inventory, zero days represent the intermittent demand pattern. Let's look at the share of zero sales days per item.

In [ ]:
zero_shares = df_mart.groupby('item_id')['demand'].apply(lambda x: (x == 0).mean()).reset_index(name='zero_fraction')

plt.figure(figsize=(8, 4))
plt.hist(zero_shares['zero_fraction'], bins=30, color='teal', edgecolor='black', alpha=0.7)
plt.title('Distribution of Zero Sales Fraction Across SKUs')
plt.xlabel('Fraction of Zero Days')
plt.ylabel('Number of SKUs')
plt.grid(axis='y', alpha=0.3)
plt.show()

## 4. EDA-Driven Modeling Decisions
1. **High Intermittency (~69% zeros overall):** Standard regression metrics can be biased towards predicting zero. We must include baseline models designed for intermittent demand (SBA/Croston) for a fair comparison.
2. **Strong Weekly Seasonality:** Weekends (Saturday/Sunday) show significantly higher sales. This justifies including `lag_7` and weekly rolling aggregates to capture weekly patterns.
3. **SNAP Events:** Walmart customers often buy more during SNAP benefit days. We must include the `snap` binary indicator as a feature.